In [1]:
import pandas as pd
import numpy as np

In [3]:
train_data=pd.read_csv("train_data.csv")
test_data=pd.read_csv("test_data.csv") 

train_days = [f'd_{i}' for i in range(1, 1914)]  # d_1 to d_1913
test_days = [f'd_{i}' for i in range(1914, 1942)]  # d_1914 to d_1941

In [4]:
# Seasonal naive: same weekday from previous week
# Since we don't have weekday info directly, we'll use 7-day lag
def seasonal_naive_forecast(train_data, test_days, forecast_horizon=28):
    """
    Forecast using sales from 7 days ago (same weekday last week)
    """
    forecasts = {}
    
    for idx, row in train_data.iterrows():
        product_id = row['id']
        # Get sales from 7 days before each test day
        forecast = []
        for i, test_day in enumerate(test_days):
            # Get the day number (e.g., 'd_1914' -> 1914)
            day_num = int(test_day.split('_')[1])
            lag_day_num = day_num - 7  # 7 days ago
            lag_day = f'd_{lag_day_num}'
            
            if lag_day in train_data.columns:
                forecast.append(row[lag_day])
            else:
                forecast.append(0)  # fallback if lag day not available
        
        forecasts[product_id] = forecast
    
    return forecasts

# Generate seasonal naive forecasts
seasonal_naive_forecasts = seasonal_naive_forecast(train_data, test_days)
print(f"Generated forecasts for {len(seasonal_naive_forecasts)} products")

Generated forecasts for 1437 products


In [7]:
def calculate_wape(actuals, forecasts):
    """
    Calculate WAPE (Weighted Absolute Percentage Error)
    WAPE = sum(|actual - forecast|) / sum(actual)
    """
    total_absolute_error = 0
    total_actual_sales = 0
    
    for product_id in forecasts.keys():
        actual = actuals[product_id]
        forecast = forecasts[product_id]
        
        for i in range(len(actual)):
            total_absolute_error += abs(actual[i] - forecast[i])
            total_actual_sales += actual[i]
    
    if total_actual_sales == 0:
        return float('inf')
    
    return total_absolute_error / total_actual_sales

# Get actual test values
actuals = {}
for idx, row in test_data.iterrows():
    product_id = row['id']
    actuals[product_id] = row[test_days].tolist()

# Calculate WAPE
seasonal_naive_wape = calculate_wape(actuals, seasonal_naive_forecasts)
print(f"Seasonal Naive WAPE: {seasonal_naive_wape:.3f} ({seasonal_naive_wape:.1%})")

Seasonal Naive WAPE: 0.944 (94.4%)
